# The 3D Visualizer

The existing {doc}`Visualizer <using-the-visualizer>` draws a liquid handler's deck. This one
draws a cartesian space and whatever is placed in it: machines, benches, and anything that
moves between them. It reads the resource tree and the tracking state each resource publishes,
so nothing about a particular machine is written into the viewer.

## Before you start

- **No hardware is needed.** The STAR here runs in simulation.
- **A browser with WebGPU**, which is Chrome, Edge and Safari 18 or newer. Firefox and older
  browsers fall back to WebGL2 and render the same scene; only GIF recording is unavailable there.
- **Nothing extra to install.** The viewer ships with PyLabRobot and runs offline.

This notebook uses `await` at the top level, which Jupyter supports directly.


## Building a facility

Two ideas are new here.

A **`Facility`** is the root: the space everything stands in, with its own origin. A machine is
placed in it at a coordinate, the same way a plate is placed on a carrier.

A **`Workcell`** is a *grouping*, not a place. It records which resources are driven together
without moving them in the tree, so a machine can belong to a workcell and still sit wherever it
physically sits. This is what lets several workcells share one facility, with transports between
them.


In [ ]:
from pylabrobot.hamilton.star.device import STARLet
from pylabrobot.resources import set_tip_tracking, set_volume_tracking
from pylabrobot.resources.coordinate import Coordinate
from pylabrobot.resources.corning import cor_96_wellplate_360uL_Fb
from pylabrobot.resources.hamilton import PLT_CAR_L5AC_A00, TIP_CAR_480_A00
from pylabrobot.resources.hamilton import hamilton_96_tiprack_1000uL
from pylabrobot.resources.resource import Resource
from pylabrobot.visualizer3D import Facility, Viewer3D, Workcell
from pylabrobot.visualizer3D.telemetry import STARTelemetry

# Tracking is what the viewer reads to colour wells and show tips, so turn it on.
set_volume_tracking(True)
set_tip_tracking(True)


In [ ]:
facility = Facility(name="facility", size_x=2600, size_y=1400, size_z=1000)
facility


Facility(name='facility', location=Coordinate(000.000, 000.000, 000.000), size_x=2600, size_y=1400, size_z=1000, category=facility)

### A machine in the facility

`STARLet(simulation=True)` gives a STAR that answers as the real instrument would, so the deck,
the channels and the X-arm all come from the machine's own configuration rather than from
anything written here.


In [ ]:
star = STARLet(simulation=True)
facility.assign_child_resource(star, location=Coordinate(0, 0, 0))
await star.setup()


2026-08-21 12:00:46,146 - pylabrobot.hamilton.star.driver.master - WARNING - the 96-head reports itself uninitialized, and there is nowhere configured to eject at. Set head96.configuration.tip_discard_location, or pass it to head96.initialize().
2026-08-21 12:00:46,154 - pylabrobot.resources.hamilton.hamilton_decks - WARNING - Resource 'pipette_channel_0' is very high on the deck: 474.7 mm. Be careful when traversing the deck.
2026-08-21 12:00:46,155 - pylabrobot.resources.hamilton.hamilton_decks - WARNING - Resource 'pipette_channel_0' is very high on the deck: 474.7 mm. Be careful when grabbing this resource.
2026-08-21 12:00:46,155 - pylabrobot.resources.hamilton.hamilton_decks - WARNING - Resource 'pipette_channel_1' is very high on the deck: 474.7 mm. Be careful when traversing the deck.
2026-08-21 12:00:46,156 - pylabrobot.resources.hamilton.hamilton_decks - WARNING - Resource 'pipette_channel_1' is very high on the deck: 474.7 mm. Be careful when grabbing this resource.
2026-08-

In [ ]:
tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
for slot in range(3):
  tip_carrier[slot] = hamilton_96_tiprack_1000uL(name=f"tips_{slot}")
star.deck.assign_child_resource(tip_carrier, rails=1)

source_carrier = PLT_CAR_L5AC_A00(name="source_carrier")
for slot in range(5):
  source_carrier[slot] = cor_96_wellplate_360uL_Fb(name=f"source_{slot}")
star.deck.assign_child_resource(source_carrier, rails=8)

destination_carrier = PLT_CAR_L5AC_A00(name="destination_carrier")
for slot in range(5):
  destination_carrier[slot] = cor_96_wellplate_360uL_Fb(name=f"destination_{slot}")
star.deck.assign_child_resource(destination_carrier, rails=14)


### Something that is not a machine

A bench has no deck, no driver and no workcell, and still takes part in the same space. This is
the case the deck-shaped viewer had no way to express.


In [ ]:
bench = Resource(name="bench", size_x=900, size_y=600, size_z=880, category="bench")
facility.assign_child_resource(bench, location=Coordinate(1300, 60, 0))
bench.assign_child_resource(
  cor_96_wellplate_360uL_Fb(name="bench_plate"), location=Coordinate(60, 60, 880)
)
bench.assign_child_resource(
  hamilton_96_tiprack_1000uL(name="bench_tips"), location=Coordinate(60, 240, 880)
)


### Recording the grouping

Nothing moves to make this true. The workcell is recorded beside the tree, and shows in the
viewer as a tag on its members rather than as a branch above them.


In [ ]:
facility.add_workcell(Workcell("star_cell", [star]))


Workcell('star_cell', 1 members)

### Two declarations the machine makes about itself

The viewer holds no constants about any instrument. Where it needs to draw something specific -
how far the channels reach across the deck, and the opening through the X-arm's carriage - the
resource declares it and the viewer draws whatever it is told.

The deck height below is a local override. A STAR deck reports `size_z` of 900 mm, which is the
working envelope rather than the deck's own extent. The honest height is the top of the arm that
rides above it. Applied here rather than in the resource library, pending a fix upstream.


In [ ]:
from pylabrobot.visualizer3D.demo import declare_channel_access

declare_channel_access(star)

DECK_HEIGHT = 334.7 + 140.0  # channel travel, plus the arm's own height
star.deck._size_z = DECK_HEIGHT
star.deck._local_size_z = DECK_HEIGHT


### Drawing the instrument from its own 3D files

A resource is drawn as a box, because a box is what its size describes. A resource can instead
declare a **mesh**, and the viewer draws that in its place.

The declaration says where the file is, what its units are, and which way is up. glTF's own
convention is metres and Y-up, so anything else has to be stated - the file does not record it.
If a file is missing the viewer says so and falls back to the box, so this is safe to run without
one.

The files are **modular**, matching how the machine is built and how the resource tree already
describes it: three bases - STARlet, STAR, STARplus - and separate modules for the autoload's tray
and sled and for the left extension housing. Each is authored in ITS OWN resource's frame, and the
manifest records where each module mounts relative to the instrument origin.

Splitting them is not tidiness. The sled is driven along x and the viewer already moves
`autoload_sled`, so a sled baked into the chassis would sit inside the moving one - the same
reason the arm is not in any of these files. The top of the hood is left off every base so the
deck reads from above, and the shell is drawn translucent for the same reason.

In [ ]:
import json
import pathlib

# Where the exports live. Prototyping path: point this at your own copy.
WEB_EXPORT = pathlib.Path(
  "/Users/camillomoschner/Documents/GitHub/bct_animations/2608_STAR/web_export"
)

# Which resource each module is drawn on. Both are things the resource tree already
# places for itself, so their meshes go on them rather than into the chassis - the
# sled because the viewer moves it, the tray because it stands in front of the
# machine rather than inside it.
MODULE_RESOURCES = {"sled": "autoload_sled", "tray": "autoload_loading_tray"}


def apply_meshes(instrument, web_export=WEB_EXPORT):
  """Give the instrument and each of its fitted modules its own mesh."""
  spec = json.loads((web_export / "manifest.json").read_text())
  frames = {name.lower(): entry for name, entry in spec["frames"].items()}
  frame = frames[str(instrument.model).lower()]
  common = {"units": spec["units"], "up": spec["up"]}

  def declare(resource, filename):
    resource.mesh = {"path": str(web_export / filename), **common}
    return filename

  applied = [declare(instrument, frame["base"])]
  by_name = {r.name: r for r in instrument.get_all_children()}
  for label, resource_name in MODULE_RESOURCES.items():
    module = frame["modules"].get(label)
    resource = by_name.get(resource_name)
    if module is not None and resource is not None:
      applied.append(declare(resource, module["file"]))

  # The left extension housing exists only when one is fitted, and is shared by all three frames.
  housing = spec.get("left_extension_housing")
  resource = by_name.get("left_extension_housing")
  if housing is not None and resource is not None:
    applied.append(declare(resource, housing["file"]))
  return applied


apply_meshes(star)


## Starting the viewer

This serves the page and opens a browser tab. The viewer stays connected and follows the state
as you run the cells below.


In [ ]:
viewer = Viewer3D(facility, telemetry=[STARTelemetry(star)], name="3d-visualizer.ipynb")
await viewer.start()


viewer on http://127.0.0.1:1338  (websocket 2122)


## Watching state change

Everything from here changes tracking state, and the viewer follows without being told to
redraw. Wells take colour from how full they are, and tip spots show whether a tip is fitted.


In [ ]:
source = star.deck.get_resource("source_0")
destination = star.deck.get_resource("destination_0")
tips = star.deck.get_resource("tips_0")

for well in source.get_all_items():
  well.tracker.set_volume(300.0)


Taking eight tips out of the rack. Watch the first column of `tips_0` empty.


In [ ]:
import asyncio

for row in "ABCDEFGH":
  spot = tips.get_item(f"{row}1")
  if spot.tracker.has_tip:
    spot.tracker.remove_tip()
  await asyncio.sleep(0.1)


Moving liquid from one plate to the other, a column at a time. Only wells that actually hold
something are moved, so no channel is asked to transfer nothing.


In [ ]:
for column in range(1, 13):
  for row in "ABCDEFGH":
    well = f"{row}{column}"
    taken = min(150.0, source.get_item(well).tracker.get_used_volume())
    if taken <= 0:
      continue
    source.get_item(well).tracker.remove_liquid(taken)
    destination.get_item(well).tracker.add_liquid(volume=taken)
  await asyncio.sleep(0.3)


## The X-arm

The arm is a resource on the deck like any other, and it moves. Its carriage is drawn
see-through with an outline, so you can read the deck underneath it, and the cyan line marks its
reference point in x.


In [ ]:
for x in (200.0, 900.0, 600.0):
  await star.x_arm.move_x(x)
  await asyncio.sleep(1.6)


In [ ]:
await star.autoload.move_x(400.0)

In [ ]:
await star.head96.move_z(320.0)

## Where a reading came from

Every telemetry reading is labelled with its provenance: `measured` when the instrument reported
it, `derived` when it follows from a reading plus a documented rule, and `unavailable` when the
machine does not publish it at all. A number the viewer cannot stand behind is shown as
unavailable rather than as a plausible value.


In [ ]:
reading = await STARTelemetry(star).read()
for channel in reading.serialize()["channels"][:3]:
  print(channel)


## A rigged mesh

The STARlet above is a single static file. A rigged file goes further: it names the parts that
move, so one mesh can follow a machine's joints instead of standing still. The PF400 below is
that case.

In [ ]:
# The rigged export this was developed against. Prototyping path: swap it for your own file
# before this notebook goes anywhere public.
PF400_GLB = "/Users/camillomoschner/Documents/GitHub/bct_animations/2606_pf400_basic_movement/web_export/pf400.glb"


### Joints, and why the viewer knows nothing about arms

A rigged file names the parts that move. The declaration maps each joint to the node that answers
to it, which axis it acts about, and whether it turns (`revolute`) or slides (`prismatic`). The
viewer moves what it is told and holds no geometry for any particular arm.

The keys are PyLabRobot's own `Axis` values for the PreciseFlex, so the same numbers that command
the arm also drive the picture. On a PF400 `Axis.BASE` is the vertical column, and the three
rotations that follow make it a SCARA.


In [ ]:
from pylabrobot.brooks.precise_flex import Axis

JOINTS = {
  str(int(Axis.BASE)): {"node": "J1_Z", "axis": "z", "type": "prismatic"},
  str(int(Axis.SHOULDER)): {"node": "J2_Shoulder", "axis": "z", "type": "revolute"},
  str(int(Axis.ELBOW)): {"node": "J3_Elbow", "axis": "z", "type": "revolute"},
  str(int(Axis.WRIST)): {"node": "J4_Wrist", "axis": "z", "type": "revolute"},
}


### A resource that publishes joint angles

The viewer follows tracking state, so an arm that wants its picture to move has to publish where
its joints are. `PreciseFlex` is a driver rather than a resource today and publishes nothing, so
this small resource stands in for that: it holds the pose and tells anyone listening when it
changes. The same shape is what the driver would need to grow.


In [ ]:
from typing import Any, Dict


class ArmResource(Resource):
  """A resource that publishes joint angles, so a rigged mesh can follow them."""

  def __init__(self, name, size_x, size_y, size_z, category="arm"):
    super().__init__(name=name, size_x=size_x, size_y=size_y, size_z=size_z, category=category)
    self.joints: Dict[str, float] = {}

  def serialize_state(self) -> Dict[str, Any]:
    return {**super().serialize_state(), "joints": dict(self.joints)}

  def set_joints(self, pose: Dict[Axis, float]) -> None:
    """Angles in degrees, travel in mm, keyed by axis."""
    self.joints.update({str(int(axis)): float(value) for axis, value in pose.items()})
    self._state_updated()


In [ ]:
# The arm's own extents. Measured from the file: 484 x 235 x 679 mm.
pf400 = ArmResource(name="pf400", size_x=484, size_y=235, size_z=679)
pf400.mesh = {"path": PF400_GLB, "units": "cm", "up": "Z", "joints": JOINTS}
facility.assign_child_resource(pf400, location=Coordinate(1400, 800, 0))


Now move it. Each pose is published as state, and the mesh follows without the viewer being told
to redraw - the same path the wells and tips took earlier.


In [ ]:
for pose in (
  {Axis.BASE: 0.0, Axis.SHOULDER: 0.0, Axis.ELBOW: 0.0, Axis.WRIST: 0.0},
  {Axis.BASE: 200.0, Axis.SHOULDER: 45.0, Axis.ELBOW: -60.0, Axis.WRIST: 20.0},
  {Axis.BASE: 350.0, Axis.SHOULDER: -30.0, Axis.ELBOW: 90.0, Axis.WRIST: -40.0},
):
  pf400.set_joints(pose)
  await asyncio.sleep(2)


Two things worth knowing about the file itself.

**Scale is not declared by glTF**, so it has to be established. Here it was cross-checked against
`pylabrobot.brooks.precise_flex.kinematics.ARM_LINKS_STANDARD`, which gives the shoulder-to-elbow
and elbow-to-wrist lengths as 225 mm and 210 mm. The file's joint origins sit 23.6 and 21.2 units
apart, so one unit is ten millimetres and this is the standard-reach arm rather than the extended
one. Node origins are not exactly kinematic centres, so expect a few percent.

**Compressed meshes need a decoder.** Files exported for the web are often Draco-compressed. The
decoder is vendored and fetched only when a compressed file actually turns up, so a viewer that
never loads one pays nothing for it.


## Stopping

The page stays open and reports itself disconnected. Re-running the start cell reconnects it.


In [ ]:
await viewer.stop()
